# Medical SLM Chunking Pipeline

This notebook handles serialization, chunking and metadata enrichment of `documents.csv` for our Domain-Specific Medical Small Language Model (SLM).

### Why 1 Row = 1 Chunk
Medical guidelines contain critical clinical rules (dosages, warnings, symptoms). Standard character-based chunking risks splitting a condition from its warning. Since each row in `documents.csv` is an atomic clinical guideline, 1 row = 1 chunk.

In [1]:

# Loading Libraries
import pandas as pd
import json
from pathlib import Path

print("Libraries loaded successfully!")

Libraries loaded successfully!


In [3]:
# Path to raw dataset
csv_file_path = "documents.csv"

# Loading data into DataFrame
df = pd.read_csv(csv_file_path)

# Verifing counts and column structure
print(f"Total documents loaded: {len(df)}")
print("\nDataset Columns:", df.columns.tolist())

# Previewing to confirm structure
df.head(1)

Total documents loaded: 24

Dataset Columns: ['document_id', 'title', 'topic', 'care_setting', 'population', 'text', 'origin', 'source_url', 'license']


,document_id,title,topic,care_setting,population,text,origin,source_url,license
0,doc_chr_001,Type 2 diabetes self-management,chronic_disease,primary_care,adult,Type 2 diabetes management combines balanced m...,synthetic,NaN,CC0-1.0


In [4]:
# Serialization & Formatting
def format_row_to_chunk(row):

    # Front-loading key metadata into text payload so vector embeddings
    # retain care setting and target population context
    formatted_text = (
        f"**Title:** {row['title']}\n"
        f"**Topic:** {row['topic']} | **Care Setting:** {row['care_setting']} | **Population:** {row['population']}\n"
        f"**Guidance:** {row['text']}"
    )

    # Extracting clean key-value pairs for metadata filtering in database
    metadata = {
        "document_id": row['document_id'],
        "title": row['title'],
        "topic": row['topic'],
        "care_setting": row['care_setting'],
        "population": row['population'],
        "origin": row['origin'],
        "license": row['license']
    }

    return {
        "id": row['document_id'],
        "text": formatted_text,
        "metadata": metadata
    }

In [5]:
# Processing all 24 Medical entries
processed_chunks = []

# Iterate over each row in the dataset
for index, row in df.iterrows():
    chunk = format_row_to_chunk(row)
    processed_chunks.append(chunk)

print(f"Successfully serialized {len(processed_chunks)} medical chunks.")

Successfully serialized 24 medical chunks.


In [6]:
# Inspecting the first formatted chunk (Chronic Disease example)
print("--- SAMPLE CHUNK 1 (Chronic Disease) ---")
print("TEXT PAYLOAD:\n", processed_chunks[0]['text'])
print("\nMETADATA:\n", json.dumps(processed_chunks[0]['metadata'], indent=2))

print("\n" + "="*50 + "\n")

# Inspecting an emergency triage chunk (Emergency example)
print("--- SAMPLE CHUNK 19 (Emergency Triage) ---")
print("TEXT PAYLOAD:\n", processed_chunks[18]['text'])
print("\nMETADATA:\n", json.dumps(processed_chunks[18]['metadata'], indent=2))

--- SAMPLE CHUNK 1 (Chronic Disease) ---
TEXT PAYLOAD:
 **Title:** Type 2 diabetes self-management
**Topic:** chronic_disease | **Care Setting:** primary_care | **Population:** adult
**Guidance:** Type 2 diabetes management combines balanced meals, regular physical activity, and adherence to prescribed medicines. Monitor blood glucose as directed and attend routine foot and eye screening. Seek urgent care for persistent vomiting, confusion, or dehydration.

METADATA:
 {
  "document_id": "doc_chr_001",
  "title": "Type 2 diabetes self-management",
  "topic": "chronic_disease",
  "care_setting": "primary_care",
  "population": "adult",
  "origin": "synthetic",
  "license": "CC0-1.0"
}


--- SAMPLE CHUNK 19 (Emergency Triage) ---
TEXT PAYLOAD:
 **Title:** Fever in infants under three months
**Topic:** emergency_triage | **Care Setting:** hospital | **Population:** infant
**Guidance:** Any documented fever in infants under three months requires urgent clinical assessment. Do not rely on ho

In [7]:
# Saveing Processed Chunks

output_file_path = "medical_chunks.json"

# Exporting to clean JSON format for vector database ingestion
with open(output_file_path, "w", encoding="utf-8") as f:
    json.dump(processed_chunks, f, indent=2)

print(f" Process Complete! Exported {len(processed_chunks)} chunks to '{output_file_path}'.")

 Process Complete! Exported 24 chunks to 'medical_chunks.json'.
